STEP 1 (DATASET UPLOAD)

In [ ]:
import pandas as pd

file_path = "movie_lines.txt"

lines = []

with open(file_path, 'r', encoding='iso-8859-1') as f:
    for line in f:
        parts = line.split(" +++$+++ ")

        if len(parts) == 5:
            character = parts[3].strip()
            dialogue = parts[4].strip()

            if len(dialogue.split()) > 3:
                lines.append([character, dialogue])

df = pd.DataFrame(lines, columns=["Character", "Dialogue"])

# Shuffle and select 2800 rows
df = df.sample(frac=1, random_state=42).reset_index(drop=True)
df = df.iloc[:2800]

df.to_csv("movie_dialogues_2800.csv", index=False)

print(df.head())
print("Total rows:", len(df))

  Character                                           Dialogue
,0     SANDY  I. I don't want to see you tomorrow. Mike's co...
,1     BETTY  Well, I just feel that life'll be much sweeter...
,2      DOUG  Look Kat it'll only take ten minutes. Without ...
,3   WILLARD                Kurtz.  I know you've heard of him.
,4  MACREADY              Could anyone have gotten it from you?
,Total rows: 2800


STEP 2(Sentiment Labeling)

In [ ]:
!pip install transformers

,Requirement already satisfied: filelock in /usr/local/lib/python3.12/dist-packages (from transformers) (3.25.2)
,Requirement already satisfied: huggingface-hub<2.0,>=1.3.0 in /usr/local/lib/python3.12/dist-packages (from transformers) (1.8.0)
,Requirement already satisfied: numpy>=1.17 in /usr/local/lib/python3.12/dist-packages (from transformers) (2.0.2)
,Requirement already satisfied: packaging>=20.0 in /usr/local/lib/python3.12/dist-packages (from transformers) (26.0)
,Requirement already satisfied: pyyaml>=5.1 in /usr/local/lib/python3.12/dist-packages (from transformers) (6.0.3)
,Requirement already satisfied: regex!=2019.12.17 in /usr/local/lib/python3.12/dist-packages (from transformers) (2025.11.3)
,Requirement already satisfied: tokenizers<=0.23.0,>=0.22.0 in /usr/local/lib/python3.12/dist-packages (from transformers) (0.22.2)
,Requirement already satisfied: typer-slim in /usr/local/lib/python3.12/dist-packages (from transformers) (0.24.0)
,Requirement already satisfied: safe

In [ ]:
from transformers import pipeline

classifier = pipeline("sentiment-analysis")

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
,Using a pipeline without specifying a model name and revision in production is not recommended.
,/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
,The secret `HF_TOKEN` does not exist in your Colab secrets.
,To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
,You will be able to reuse this secret in all of your notebooks.
,Please note that authentication is recommended but still optional to access public models or datasets.
,  warnings.warn(


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

,WARNING:huggingface_hub.utils._http:Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

In [ ]:
import pandas as pd

df = pd.read_csv("movie_dialogues_2800.csv")
df.head()

,Character,Dialogue
0,SANDY,I. I don't want to see you tomorrow. Mike's co...
1,BETTY,"Well, I just feel that life'll be much sweeter..."
2,DOUG,Look Kat it'll only take ten minutes. Without ...
3,WILLARD,Kurtz. I know you've heard of him.
4,MACREADY,Could anyone have gotten it from you?


In [ ]:
def get_sentiment(text):
    result = classifier(text[:512])[0]   # limit for safety

    if result['label'] == 'POSITIVE':
        return "Positive"
    elif result['label'] == 'NEGATIVE':
        return "Negative"
    else:
        return "Neutral"

df['Sentiment'] = df['Dialogue'].apply(get_sentiment)

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


In [ ]:
df.to_csv("movie_dialogues_with_sentiment.csv", index=False)

In [ ]:
from google.colab import files
files.download("movie_dialogues_with_sentiment.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

step 3(Train-Test Split + Data Preparation)

In [ ]:
import pandas as pd

df = pd.read_csv("movie_dialogues_with_sentiment.csv")
df.head()

,Character,Dialogue,Sentiment
0,SANDY,I. I don't want to see you tomorrow. Mike's co...,Positive
1,BETTY,"Well, I just feel that life'll be much sweeter...",Positive
2,DOUG,Look Kat it'll only take ten minutes. Without ...,Negative
3,WILLARD,Kurtz. I know you've heard of him.,Positive
4,MACREADY,Could anyone have gotten it from you?,Negative


In [ ]:
label_map = {
    "Negative": 0,
    "Neutral": 1,
    "Positive": 2
}

df['Label'] = df['Sentiment'].map(label_map)

In [ ]:
X = df['Dialogue']
y = df['Label']

In [ ]:
from sklearn.model_selection import train_test_split

X_train_80, X_test_80, y_train_80, y_test_80 = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
X_train_90, X_test_90, y_train_90, y_test_90 = train_test_split(
    X, y, test_size=0.1, random_state=42
)

In [ ]:
print("80:20 Split")
print("Train:", len(X_train_80), "Test:", len(X_test_80))

print("\n90:10 Split")
print("Train:", len(X_train_90), "Test:", len(X_test_90))

80:20 Split
,Train: 2240 Test: 560
,
,90:10 Split
,Train: 2520 Test: 280


step 4(Text Preprocessing)

In [ ]:
!pip install transformers torch scikit-learn

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

,Requirement already satisfied: torch in /usr/local/lib/python3.12/dist-packages (2.10.0+cu128)
,Requirement already satisfied: scikit-learn in /usr/local/lib/python3.12/dist-packages (1.6.1)
,Requirement already satisfied: filelock in /usr/local/lib/python3.12/dist-packages (from transformers) (3.25.2)
,Requirement already satisfied: huggingface-hub<2.0,>=1.3.0 in /usr/local/lib/python3.12/dist-packages (from transformers) (1.8.0)
,Requirement already satisfied: numpy>=1.17 in /usr/local/lib/python3.12/dist-packages (from transformers) (2.0.2)
,Requirement already satisfied: packaging>=20.0 in /usr/local/lib/python3.12/dist-packages (from transformers) (26.0)
,Requirement already satisfied: pyyaml>=5.1 in /usr/local/lib/python3.12/dist-packages (from transformers) (6.0.3)
,Requirement already satisfied: regex!=2019.12.17 in /usr/local/lib/python3.12/dist-packages (from transformers) (2025.11.3)
,Requirement already satisfied: tokenizers<=0.23.0,>=0.22.0 in /usr/local/lib/python3.12/di

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
X_train_tokens = tokenizer(
    list(X_train_80),
    padding=True,
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

X_test_tokens = tokenizer(
    list(X_test_80),
    padding=True,
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

In [ ]:
class Dataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

In [ ]:
train_dataset = Dataset(X_train_tokens, list(y_train_80))
test_dataset = Dataset(X_test_tokens, list(y_test_80))

step 5(training transformer model)

In [ ]:
import torch
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments

In [ ]:
df = pd.read_csv("movie_dialogues_with_sentiment.csv")
df.head()

,Character,Dialogue,Sentiment
0,SANDY,I. I don't want to see you tomorrow. Mike's co...,Positive
1,BETTY,"Well, I just feel that life'll be much sweeter...",Positive
2,DOUG,Look Kat it'll only take ten minutes. Without ...,Negative
3,WILLARD,Kurtz. I know you've heard of him.,Positive
4,MACREADY,Could anyone have gotten it from you?,Negative


BERT MODEL

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=3
)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
,Key                                        | Status     | 
,-------------------------------------------+------------+-
,cls.seq_relationship.bias                  | UNEXPECTED | 
,cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
,cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
,cls.seq_relationship.weight                | UNEXPECTED | 
,cls.predictions.transform.dense.weight     | UNEXPECTED | 
,cls.predictions.transform.dense.bias       | UNEXPECTED | 
,cls.predictions.bias                       | UNEXPECTED | 
,classifier.bias                            | MISSING    | 
,classifier.weight                          | MISSING    | 
,
,Notes:
,- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
,- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
def compute_metrics(pred):
    logits, labels = pred
    preds = logits.argmax(axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='weighted')
    acc = accuracy_score(labels, preds)

    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

BERT (3 EPOCHS)

In [ ]:
training_args = TrainingArguments(
    output_dir="./bert_results_3",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    logging_dir="./bert_logs_3",
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()

Step,Training Loss
500,0.385940


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=840, training_loss=0.26527806236630397, metrics={'train_runtime': 179.724, 'train_samples_per_second': 37.391, 'train_steps_per_second': 4.674, 'total_flos': 442030541783040.0, 'train_loss': 0.26527806236630397, 'epoch': 3.0})

In [ ]:
results_bert_3 = trainer.evaluate()
print("BERT (3 epochs):", results_bert_3)

BERT (3 epochs): {'eval_loss': 0.7935815453529358, 'eval_accuracy': 0.8535714285714285, 'eval_f1': 0.8532504306070253, 'eval_precision': 0.8532845307385184, 'eval_recall': 0.8535714285714285, 'eval_runtime': 3.9857, 'eval_samples_per_second': 140.502, 'eval_steps_per_second': 17.563, 'epoch': 3.0}


BERT (5 EPOCHS)

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=3
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
,Key                                        | Status     | 
,-------------------------------------------+------------+-
,cls.seq_relationship.bias                  | UNEXPECTED | 
,cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
,cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
,cls.seq_relationship.weight                | UNEXPECTED | 
,cls.predictions.transform.dense.weight     | UNEXPECTED | 
,cls.predictions.transform.dense.bias       | UNEXPECTED | 
,cls.predictions.bias                       | UNEXPECTED | 
,classifier.bias                            | MISSING    | 
,classifier.weight                          | MISSING    | 
,
,Notes:
,- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
,- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
training_args = TrainingArguments(
    output_dir="./bert_results_5",
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    logging_dir="./bert_logs_5",
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

results_bert_5 = trainer.evaluate()
print("BERT (5 epochs):", results_bert_5)

Step,Training Loss
500,0.385534
1000,0.093003


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

BERT (5 epochs): {'eval_loss': 1.081394910812378, 'eval_accuracy': 0.8321428571428572, 'eval_f1': 0.8310840442148264, 'eval_precision': 0.8322132995209919, 'eval_recall': 0.8321428571428572, 'eval_runtime': 3.9947, 'eval_samples_per_second': 140.185, 'eval_steps_per_second': 17.523, 'epoch': 5.0}


BERT (7 EPOCHS)

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=3
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
,Key                                        | Status     | 
,-------------------------------------------+------------+-
,cls.seq_relationship.bias                  | UNEXPECTED | 
,cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
,cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
,cls.seq_relationship.weight                | UNEXPECTED | 
,cls.predictions.transform.dense.weight     | UNEXPECTED | 
,cls.predictions.transform.dense.bias       | UNEXPECTED | 
,cls.predictions.bias                       | UNEXPECTED | 
,classifier.bias                            | MISSING    | 
,classifier.weight                          | MISSING    | 
,
,Notes:
,- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
,- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
training_args = TrainingArguments(
    output_dir="./bert_results_7",
    num_train_epochs=7,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    logging_dir="./bert_logs_7",
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

results_bert_7 = trainer.evaluate()
print("BERT (7 epochs):", results_bert_7)

Step,Training Loss
500,0.228897
1000,0.067941
1500,0.012243


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

BERT (7 epochs): {'eval_loss': 1.185781717300415, 'eval_accuracy': 0.8589285714285714, 'eval_f1': 0.8587424910655835, 'eval_precision': 0.8586956501454679, 'eval_recall': 0.8589285714285714, 'eval_runtime': 4.6086, 'eval_samples_per_second': 121.513, 'eval_steps_per_second': 15.189, 'epoch': 7.0}


RoBERTa MODEL

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=3
)

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
,Key                             | Status     | 
,--------------------------------+------------+-
,lm_head.layer_norm.bias         | UNEXPECTED | 
,lm_head.dense.bias              | UNEXPECTED | 
,roberta.embeddings.position_ids | UNEXPECTED | 
,lm_head.dense.weight            | UNEXPECTED | 
,lm_head.layer_norm.weight       | UNEXPECTED | 
,lm_head.bias                    | UNEXPECTED | 
,classifier.out_proj.weight      | MISSING    | 
,classifier.dense.bias           | MISSING    | 
,classifier.dense.weight         | MISSING    | 
,classifier.out_proj.bias        | MISSING    | 
,
,Notes:
,- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
,- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("roberta-base")

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
X_train_tokens = tokenizer(
    list(X_train_80),
    padding=True,
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

X_test_tokens = tokenizer(
    list(X_test_80),
    padding=True,
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

In [ ]:
train_dataset = Dataset(X_train_tokens, list(y_train_80))
test_dataset = Dataset(X_test_tokens, list(y_test_80))

RoBERTa (3 epochs)

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./roberta_results_3",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    logging_dir="./roberta_logs_3",
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)
trainer.train()
results_roberta_3 = trainer.evaluate()
print("RoBERTa (3 epochs):", results_roberta_3)

Step,Training Loss
500,0.677256


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

RoBERTa (3 epochs): {'eval_loss': 0.572613000869751, 'eval_accuracy': 0.7660714285714286, 'eval_f1': 0.7635033713039411, 'eval_precision': 0.7658015550765995, 'eval_recall': 0.7660714285714286, 'eval_runtime': 3.7426, 'eval_samples_per_second': 149.628, 'eval_steps_per_second': 18.704, 'epoch': 3.0}


RoBERTa (5 epochs)

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=3
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
,Key                             | Status     | 
,--------------------------------+------------+-
,lm_head.layer_norm.bias         | UNEXPECTED | 
,lm_head.dense.bias              | UNEXPECTED | 
,roberta.embeddings.position_ids | UNEXPECTED | 
,lm_head.dense.weight            | UNEXPECTED | 
,lm_head.layer_norm.weight       | UNEXPECTED | 
,lm_head.bias                    | UNEXPECTED | 
,classifier.out_proj.weight      | MISSING    | 
,classifier.dense.bias           | MISSING    | 
,classifier.dense.weight         | MISSING    | 
,classifier.out_proj.bias        | MISSING    | 
,
,Notes:
,- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
,- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./roberta_results_5",
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    logging_dir="./roberta_logs_5",
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)
trainer.train()
results_roberta_5 = trainer.evaluate()
print("RoBERTa (5 epochs):", results_roberta_5)

Step,Training Loss
500,0.565869
1000,0.350123


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

RoBERTa (5 epochs): {'eval_loss': 0.8138005137443542, 'eval_accuracy': 0.8339285714285715, 'eval_f1': 0.8329443838604144, 'eval_precision': 0.8339371493963583, 'eval_recall': 0.8339285714285715, 'eval_runtime': 3.7755, 'eval_samples_per_second': 148.323, 'eval_steps_per_second': 18.54, 'epoch': 5.0}


RoBERTa (7 epochs)

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=3
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
,Key                             | Status     | 
,--------------------------------+------------+-
,lm_head.layer_norm.bias         | UNEXPECTED | 
,lm_head.dense.bias              | UNEXPECTED | 
,roberta.embeddings.position_ids | UNEXPECTED | 
,lm_head.dense.weight            | UNEXPECTED | 
,lm_head.layer_norm.weight       | UNEXPECTED | 
,lm_head.bias                    | UNEXPECTED | 
,classifier.out_proj.weight      | MISSING    | 
,classifier.dense.bias           | MISSING    | 
,classifier.dense.weight         | MISSING    | 
,classifier.out_proj.bias        | MISSING    | 
,
,Notes:
,- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
,- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./roberta_results_7",
    num_train_epochs=7,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    logging_dir="./roberta_logs_7",
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)
trainer.train()
results_roberta_7 = trainer.evaluate()
print("RoBERTa (7 epochs):", results_roberta_7)

Step,Training Loss
500,0.559251
1000,0.341036
1500,0.160808


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

RoBERTa (7 epochs): {'eval_loss': 0.9980577826499939, 'eval_accuracy': 0.8285714285714286, 'eval_f1': 0.8274900877087589, 'eval_precision': 0.8285866593558902, 'eval_recall': 0.8285714285714286, 'eval_runtime': 3.7563, 'eval_samples_per_second': 149.084, 'eval_steps_per_second': 18.635, 'epoch': 7.0}


DeBERTa MODEL

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "microsoft/deberta-base",
    num_labels=3
)

config.json:   0%|          | 0.00/474 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/559M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/559M [00:00<?, ?B/s]

DebertaForSequenceClassification LOAD REPORT from: microsoft/deberta-base
,Key                                     | Status     | 
,----------------------------------------+------------+-
,lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
,lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
,lm_predictions.lm_head.bias             | UNEXPECTED | 
,lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
,lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
,classifier.bias                         | MISSING    | 
,classifier.weight                       | MISSING    | 
,pooler.dense.bias                       | MISSING    | 
,pooler.dense.weight                     | MISSING    | 
,
,Notes:
,- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
,- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-base")

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

In [ ]:
X_train_tokens = tokenizer(
    list(X_train_80),
    padding=True,
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

X_test_tokens = tokenizer(
    list(X_test_80),
    padding=True,
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

In [ ]:
train_dataset = Dataset(X_train_tokens, list(y_train_80))
test_dataset = Dataset(X_test_tokens, list(y_test_80))

DeBERTa (3 epochs)

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./deberta_results_3",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    logging_dir="./deberta_logs_3",
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)
trainer.train()
results_deberta_3 = trainer.evaluate()
print("DeBERTa (3 epochs):", results_deberta_3)

Step,Training Loss
500,0.481943


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

DeBERTa (3 epochs): {'eval_loss': 0.7517894506454468, 'eval_accuracy': 0.8464285714285714, 'eval_f1': 0.8454598702390965, 'eval_precision': 0.8467198601813987, 'eval_recall': 0.8464285714285714, 'eval_runtime': 5.1072, 'eval_samples_per_second': 109.649, 'eval_steps_per_second': 13.706, 'epoch': 3.0}


DeBERTa (5 epochs)

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "microsoft/deberta-base",
    num_labels=3
)

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

DebertaForSequenceClassification LOAD REPORT from: microsoft/deberta-base
,Key                                     | Status     | 
,----------------------------------------+------------+-
,lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
,lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
,lm_predictions.lm_head.bias             | UNEXPECTED | 
,lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
,lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
,classifier.bias                         | MISSING    | 
,classifier.weight                       | MISSING    | 
,pooler.dense.bias                       | MISSING    | 
,pooler.dense.weight                     | MISSING    | 
,
,Notes:
,- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
,- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./deberta_results_5",
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    logging_dir="./deberta_logs_5",
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

results_deberta_5 = trainer.evaluate()
print("DeBERTa (5 epochs):", results_deberta_5)

Step,Training Loss
500,0.496620
1000,0.212748


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

DeBERTa (5 epochs): {'eval_loss': 1.0320383310317993, 'eval_accuracy': 0.8303571428571429, 'eval_f1': 0.8294776397332217, 'eval_precision': 0.8301698649609097, 'eval_recall': 0.8303571428571429, 'eval_runtime': 5.1646, 'eval_samples_per_second': 108.43, 'eval_steps_per_second': 13.554, 'epoch': 5.0}


DeBERTa (7 epochs)

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "microsoft/deberta-base",
    num_labels=3
)

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

DebertaForSequenceClassification LOAD REPORT from: microsoft/deberta-base
,Key                                     | Status     | 
,----------------------------------------+------------+-
,lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
,lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
,lm_predictions.lm_head.bias             | UNEXPECTED | 
,lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
,lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
,classifier.bias                         | MISSING    | 
,classifier.weight                       | MISSING    | 
,pooler.dense.bias                       | MISSING    | 
,pooler.dense.weight                     | MISSING    | 
,
,Notes:
,- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
,- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./deberta_results_7",
    num_train_epochs=7,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    logging_dir="./deberta_logs_7",
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

results_deberta_7 = trainer.evaluate()
print("DeBERTa (7 epochs):", results_deberta_7)

Step,Training Loss
500,0.468337
1000,0.192981
1500,0.041832


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

DeBERTa (7 epochs): {'eval_loss': 1.3357486724853516, 'eval_accuracy': 0.8232142857142857, 'eval_f1': 0.8226614530647455, 'eval_precision': 0.8227565494282818, 'eval_recall': 0.8232142857142857, 'eval_runtime': 5.0808, 'eval_samples_per_second': 110.219, 'eval_steps_per_second': 13.777, 'epoch': 7.0}


DistilBERT MODEL

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=3
)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
,Key                     | Status     | 
,------------------------+------------+-
,vocab_projector.bias    | UNEXPECTED | 
,vocab_transform.bias    | UNEXPECTED | 
,vocab_transform.weight  | UNEXPECTED | 
,vocab_layer_norm.bias   | UNEXPECTED | 
,vocab_layer_norm.weight | UNEXPECTED | 
,classifier.bias         | MISSING    | 
,classifier.weight       | MISSING    | 
,pre_classifier.weight   | MISSING    | 
,pre_classifier.bias     | MISSING    | 
,
,Notes:
,- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
,- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
X_train_tokens = tokenizer(list(X_train_80), padding=True, truncation=True, max_length=128, return_tensors="pt")
X_test_tokens = tokenizer(list(X_test_80), padding=True, truncation=True, max_length=128, return_tensors="pt")

In [ ]:
train_dataset = Dataset(X_train_tokens, list(y_train_80))
test_dataset = Dataset(X_test_tokens, list(y_test_80))

DistilBERT (3 Epochs)

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./distilbert_results_3",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    logging_dir="./distilbert_logs_3",
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

results_distilbert_3 = trainer.evaluate()
print("DistilBERT (3 epochs):", results_distilbert_3)

Step,Training Loss
500,0.366309


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

DistilBERT (3 epochs): {'eval_loss': 0.7527303695678711, 'eval_accuracy': 0.8482142857142857, 'eval_f1': 0.8476397743103994, 'eval_precision': 0.8480014342443267, 'eval_recall': 0.8482142857142857, 'eval_runtime': 2.0533, 'eval_samples_per_second': 272.728, 'eval_steps_per_second': 34.091, 'epoch': 3.0}


DistilBERT (5 Epochs)

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=3
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
,Key                     | Status     | 
,------------------------+------------+-
,vocab_projector.bias    | UNEXPECTED | 
,vocab_transform.bias    | UNEXPECTED | 
,vocab_transform.weight  | UNEXPECTED | 
,vocab_layer_norm.bias   | UNEXPECTED | 
,vocab_layer_norm.weight | UNEXPECTED | 
,classifier.bias         | MISSING    | 
,classifier.weight       | MISSING    | 
,pre_classifier.weight   | MISSING    | 
,pre_classifier.bias     | MISSING    | 
,
,Notes:
,- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
,- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./distilbert_results_5",
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    logging_dir="./distilbert_logs_5",
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

results_distilbert_5 = trainer.evaluate()
print("DistilBERT (5 epochs):", results_distilbert_5)

Step,Training Loss
500,0.370412
1000,0.076626


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

DistilBERT (5 epochs): {'eval_loss': 0.9941375851631165, 'eval_accuracy': 0.8410714285714286, 'eval_f1': 0.8402474730132289, 'eval_precision': 0.8410170575692963, 'eval_recall': 0.8410714285714286, 'eval_runtime': 1.9969, 'eval_samples_per_second': 280.432, 'eval_steps_per_second': 35.054, 'epoch': 5.0}


DistilBERT (7 Epochs)

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=3
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
,Key                     | Status     | 
,------------------------+------------+-
,vocab_projector.bias    | UNEXPECTED | 
,vocab_transform.bias    | UNEXPECTED | 
,vocab_transform.weight  | UNEXPECTED | 
,vocab_layer_norm.bias   | UNEXPECTED | 
,vocab_layer_norm.weight | UNEXPECTED | 
,classifier.bias         | MISSING    | 
,classifier.weight       | MISSING    | 
,pre_classifier.weight   | MISSING    | 
,pre_classifier.bias     | MISSING    | 
,
,Notes:
,- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
,- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./distilbert_results_7",
    num_train_epochs=7,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    logging_dir="./distilbert_logs_7",
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

results_distilbert_7 = trainer.evaluate()
print("DistilBERT (7 epochs):", results_distilbert_7)

Step,Training Loss
500,0.383515
1000,0.111887
1500,0.013013


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

DistilBERT (7 epochs): {'eval_loss': 1.0758122205734253, 'eval_accuracy': 0.85, 'eval_f1': 0.848139303482587, 'eval_precision': 0.8528791520979021, 'eval_recall': 0.85, 'eval_runtime': 2.0165, 'eval_samples_per_second': 277.711, 'eval_steps_per_second': 34.714, 'epoch': 7.0}


ELECTRA

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    "google/electra-base-discriminator",
    num_labels=3
)

config.json:   0%|          | 0.00/666 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

ElectraForSequenceClassification LOAD REPORT from: google/electra-base-discriminator
,Key                                               | Status     | 
,--------------------------------------------------+------------+-
,discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
,discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
,discriminator_predictions.dense.weight            | UNEXPECTED | 
,electra.embeddings_project.weight                 | UNEXPECTED | 
,discriminator_predictions.dense.bias              | UNEXPECTED | 
,electra.embeddings_project.bias                   | UNEXPECTED | 
,classifier.out_proj.weight                        | MISSING    | 
,classifier.dense.bias                             | MISSING    | 
,classifier.dense.weight                           | MISSING    | 
,classifier.out_proj.bias                          | MISSING    | 
,
,Notes:
,- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect ide

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("google/electra-base-discriminator")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
X_train_tokens = tokenizer(list(X_train_80), padding=True, truncation=True, max_length=128, return_tensors="pt")
X_test_tokens = tokenizer(list(X_test_80), padding=True, truncation=True, max_length=128, return_tensors="pt")

In [ ]:
train_dataset = Dataset(X_train_tokens, list(y_train_80))
test_dataset = Dataset(X_test_tokens, list(y_test_80))

ELECTRA (3 Epochs)

In [ ]:
training_args = TrainingArguments(
    output_dir="./electra_results_3",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    logging_dir="./electra_logs_3",
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

results_electra_3 = trainer.evaluate()
print("ELECTRA (3 epochs):", results_electra_3)

Step,Training Loss
500,0.466810


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

ELECTRA (3 epochs): {'eval_loss': 0.6253449320793152, 'eval_accuracy': 0.8446428571428571, 'eval_f1': 0.8433499168704296, 'eval_precision': 0.8455539358600583, 'eval_recall': 0.8446428571428571, 'eval_runtime': 3.753, 'eval_samples_per_second': 149.213, 'eval_steps_per_second': 18.652, 'epoch': 3.0}


ELECTRA (5 Epochs)

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "google/electra-base-discriminator",
    num_labels=3
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

ElectraForSequenceClassification LOAD REPORT from: google/electra-base-discriminator
,Key                                               | Status     | 
,--------------------------------------------------+------------+-
,discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
,discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
,discriminator_predictions.dense.weight            | UNEXPECTED | 
,electra.embeddings_project.weight                 | UNEXPECTED | 
,discriminator_predictions.dense.bias              | UNEXPECTED | 
,electra.embeddings_project.bias                   | UNEXPECTED | 
,classifier.out_proj.weight                        | MISSING    | 
,classifier.dense.bias                             | MISSING    | 
,classifier.dense.weight                           | MISSING    | 
,classifier.out_proj.bias                          | MISSING    | 
,
,Notes:
,- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect ide

In [ ]:
training_args = TrainingArguments(
    output_dir="./electra_results_5",
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    logging_dir="./electra_logs_5",
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

results_electra_5 = trainer.evaluate()
print("ELECTRA (5 epochs):", results_electra_5)

Step,Training Loss
500,0.507280
1000,0.290328


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

ELECTRA (5 epochs): {'eval_loss': 0.7590488195419312, 'eval_accuracy': 0.8392857142857143, 'eval_f1': 0.8392857142857143, 'eval_precision': 0.8392857142857143, 'eval_recall': 0.8392857142857143, 'eval_runtime': 3.7134, 'eval_samples_per_second': 150.805, 'eval_steps_per_second': 18.851, 'epoch': 5.0}


ELECTRA (7 Epochs)

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    "google/electra-base-discriminator",
    num_labels=3
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

ElectraForSequenceClassification LOAD REPORT from: google/electra-base-discriminator
,Key                                               | Status     | 
,--------------------------------------------------+------------+-
,discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
,discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
,discriminator_predictions.dense.weight            | UNEXPECTED | 
,electra.embeddings_project.weight                 | UNEXPECTED | 
,discriminator_predictions.dense.bias              | UNEXPECTED | 
,electra.embeddings_project.bias                   | UNEXPECTED | 
,classifier.out_proj.weight                        | MISSING    | 
,classifier.dense.bias                             | MISSING    | 
,classifier.dense.weight                           | MISSING    | 
,classifier.out_proj.bias                          | MISSING    | 
,
,Notes:
,- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect ide

In [ ]:
training_args = TrainingArguments(
    output_dir="./electra_results_7",
    num_train_epochs=7,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    logging_dir="./electra_logs_7",
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

results_electra_7 = trainer.evaluate()
print("ELECTRA (7 epochs):", results_electra_7)

Step,Training Loss
500,0.445873
1000,0.197191
1500,0.055659


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

ELECTRA (7 epochs): {'eval_loss': 1.0904022455215454, 'eval_accuracy': 0.8464285714285714, 'eval_f1': 0.8465719769958202, 'eval_precision': 0.8468025255775761, 'eval_recall': 0.8464285714285714, 'eval_runtime': 3.7521, 'eval_samples_per_second': 149.248, 'eval_steps_per_second': 18.656, 'epoch': 7.0}


SVM KERNELS

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=5000)

X_train_tfidf_80 = vectorizer.fit_transform(X_train_80)
X_test_tfidf_80 = vectorizer.transform(X_test_80)

In [ ]:
from sklearn.svm import SVC

LINEAR KERNEL

In [ ]:
svm_linear = SVC(kernel='linear')
svm_linear.fit(X_train_tfidf_80, y_train_80)
y_pred_linear = svm_linear.predict(X_test_tfidf_80)

POLYNOMIAL KERNEL

In [ ]:
svm_poly = SVC(kernel='poly')
svm_poly.fit(X_train_tfidf_80, y_train_80)
y_pred_poly = svm_poly.predict(X_test_tfidf_80)

RBF (Gaussian) Kernel

In [ ]:
svm_rbf = SVC(kernel='rbf')

svm_rbf.fit(X_train_tfidf_80, y_train_80)

y_pred_rbf = svm_rbf.predict(X_test_tfidf_80)

EVALUATION

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

LINEAR RESULTS

In [ ]:
print("Linear SVM")
print("Accuracy:", accuracy_score(y_test_80, y_pred_linear))
print("Precision:", precision_score(y_test_80, y_pred_linear, average='weighted'))
print("Recall:", recall_score(y_test_80, y_pred_linear, average='weighted'))
print("F1 Score:", f1_score(y_test_80, y_pred_linear, average='weighted'))

Linear SVM
,Accuracy: 0.7178571428571429
,Precision: 0.7226072027948078
,Recall: 0.7178571428571429
,F1 Score: 0.7079917379769579


POLYNOMIAL RESULTS

In [ ]:
print("Polynomial SVM")
print("Accuracy:", accuracy_score(y_test_80, y_pred_poly))
print("Precision:", precision_score(y_test_80, y_pred_poly, average='weighted'))
print("Recall:", recall_score(y_test_80, y_pred_poly, average='weighted'))
print("F1 Score:", f1_score(y_test_80, y_pred_poly, average='weighted'))

Polynomial SVM
,Accuracy: 0.6017857142857143
,Precision: 0.7146315086782375
,Recall: 0.6017857142857143
,F1 Score: 0.4906166634363217


RBF RESULTS

In [ ]:
print("RBF SVM")
print("Accuracy:", accuracy_score(y_test_80, y_pred_rbf))
print("Precision:", precision_score(y_test_80, y_pred_rbf, average='weighted'))
print("Recall:", recall_score(y_test_80, y_pred_rbf, average='weighted'))
print("F1 Score:", f1_score(y_test_80, y_pred_rbf, average='weighted'))

RBF SVM
,Accuracy: 0.6982142857142857
,Precision: 0.7234394088669951
,Recall: 0.6982142857142857
,F1 Score: 0.6735549820957509


VOTING CLASSIFIER

In [ ]:
from sklearn.ensemble import VotingClassifier

voting_clf = VotingClassifier(
    estimators=[
        ('linear', svm_linear),
        ('poly', svm_poly),
        ('rbf', svm_rbf)
    ],
    voting='hard'
)

voting_clf.fit(X_train_tfidf_80, y_train_80)

y_pred_vote = voting_clf.predict(X_test_tfidf_80)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

print("Voting Classifier")
print("Accuracy:", accuracy_score(y_test_80, y_pred_vote))
print("Precision:", precision_score(y_test_80, y_pred_vote, average='weighted'))
print("Recall:", recall_score(y_test_80, y_pred_vote, average='weighted'))
print("F1 Score:", f1_score(y_test_80, y_pred_vote, average='weighted'))

Voting Classifier
,Accuracy: 0.6928571428571428
,Precision: 0.7186182669789226
,Recall: 0.6928571428571428
,F1 Score: 0.6666056166056166


CONFUSION MATRIX

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test_80, y_pred_vote)
print("Confusion Matrix:\n", cm)

Confusion Matrix:
, [[292  26]
, [146  96]]


MCC (Matthews Correlation Coefficient)

In [ ]:
from sklearn.metrics import matthews_corrcoef

mcc = matthews_corrcoef(y_test_80, y_pred_vote)
print("MCC:", mcc)

MCC: 0.37793988819094276


RMSE (Root Mean Square Error)

In [ ]:
from sklearn.metrics import mean_squared_error
import numpy as np

rmse = np.sqrt(mean_squared_error(y_test_80, y_pred_vote))
print("RMSE:", rmse)

RMSE: 1.1084094137869043


RESULTS

In [ ]:
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

data = {
    "Model": [
        "BERT (3)", "BERT (5)", "BERT (7)",
        "RoBERTa (3)", "RoBERTa (5)", "RoBERTa (7)",
        "DeBERTa (3)", "DeBERTa (5)", "DeBERTa (7)",
        "DistilBERT (3)",
        "ELECTRA (3)", "ELECTRA (5)", "ELECTRA (7)",
        "SVM Linear", "SVM Poly", "SVM RBF", "Voting"
    ],

    "Accuracy": [
        results_bert_3['eval_accuracy'],
        results_bert_5['eval_accuracy'],
        results_bert_7['eval_accuracy'],

        results_roberta_3['eval_accuracy'],
        results_roberta_5['eval_accuracy'],
        results_roberta_7['eval_accuracy'],

        results_deberta_3['eval_accuracy'],
        results_deberta_5['eval_accuracy'],
        results_deberta_7['eval_accuracy'],

        results_distilbert_3['eval_accuracy'],

        results_electra_3['eval_accuracy'],
        results_electra_5['eval_accuracy'],
        results_electra_7['eval_accuracy'],

        accuracy_score(y_test_80, y_pred_linear),
        accuracy_score(y_test_80, y_pred_poly),
        accuracy_score(y_test_80, y_pred_rbf),
        accuracy_score(y_test_80, y_pred_vote)
    ],

    "Precision": [
        results_bert_3['eval_precision'],
        results_bert_5['eval_precision'],
        results_bert_7['eval_precision'],

        results_roberta_3['eval_precision'],
        results_roberta_5['eval_precision'],
        results_roberta_7['eval_precision'],

        results_deberta_3['eval_precision'],
        results_deberta_5['eval_precision'],
        results_deberta_7['eval_precision'],

        results_distilbert_3['eval_precision'],

        results_electra_3['eval_precision'],
        results_electra_5['eval_precision'],
        results_electra_7['eval_precision'],

        precision_score(y_test_80, y_pred_linear, average='weighted'),
        precision_score(y_test_80, y_pred_poly, average='weighted'),
        precision_score(y_test_80, y_pred_rbf, average='weighted'),
        precision_score(y_test_80, y_pred_vote, average='weighted')
    ],

    "Recall": [
        results_bert_3['eval_recall'],
        results_bert_5['eval_recall'],
        results_bert_7['eval_recall'],

        results_roberta_3['eval_recall'],
        results_roberta_5['eval_recall'],
        results_roberta_7['eval_recall'],

        results_deberta_3['eval_recall'],
        results_deberta_5['eval_recall'],
        results_deberta_7['eval_recall'],

        results_distilbert_3['eval_recall'],

        results_electra_3['eval_recall'],
        results_electra_5['eval_recall'],
        results_electra_7['eval_recall'],

        recall_score(y_test_80, y_pred_linear, average='weighted'),
        recall_score(y_test_80, y_pred_poly, average='weighted'),
        recall_score(y_test_80, y_pred_rbf, average='weighted'),
        recall_score(y_test_80, y_pred_vote, average='weighted')
    ],

    "F1 Score": [
        results_bert_3['eval_f1'],
        results_bert_5['eval_f1'],
        results_bert_7['eval_f1'],

        results_roberta_3['eval_f1'],
        results_roberta_5['eval_f1'],
        results_roberta_7['eval_f1'],

        results_deberta_3['eval_f1'],
        results_deberta_5['eval_f1'],
        results_deberta_7['eval_f1'],

        results_distilbert_3['eval_f1'],

        results_electra_3['eval_f1'],
        results_electra_5['eval_f1'],
        results_electra_7['eval_f1'],

        f1_score(y_test_80, y_pred_linear, average='weighted'),
        f1_score(y_test_80, y_pred_poly, average='weighted'),
        f1_score(y_test_80, y_pred_rbf, average='weighted'),
        f1_score(y_test_80, y_pred_vote, average='weighted')
    ]
}

df_results = pd.DataFrame(data)
print(df_results)

             Model  Accuracy  Precision    Recall  F1 Score
,0         BERT (3)  0.853571   0.853285  0.853571  0.853250
,1         BERT (5)  0.832143   0.832213  0.832143  0.831084
,2         BERT (7)  0.858929   0.858696  0.858929  0.858742
,3      RoBERTa (3)  0.766071   0.765802  0.766071  0.763503
,4      RoBERTa (5)  0.833929   0.833937  0.833929  0.832944
,5      RoBERTa (7)  0.828571   0.828587  0.828571  0.827490
,6      DeBERTa (3)  0.846429   0.846720  0.846429  0.845460
,7      DeBERTa (5)  0.830357   0.830170  0.830357  0.829478
,8      DeBERTa (7)  0.823214   0.822757  0.823214  0.822661
,9   DistilBERT (3)  0.848214   0.848001  0.848214  0.847640
,10     ELECTRA (3)  0.844643   0.845554  0.844643  0.843350
,11     ELECTRA (5)  0.839286   0.839286  0.839286  0.839286
,12     ELECTRA (7)  0.846429   0.846803  0.846429  0.846572
,13      SVM Linear  0.717857   0.722607  0.717857  0.707992
,14        SVM Poly  0.601786   0.714632  0.601786  0.490617
,15         SVM RBF  0.69

In [ ]:
df_results.round(3)

,Model,Accuracy,Precision,Recall,F1 Score
0,BERT (3),0.854,0.853,0.854,0.853
1,BERT (5),0.832,0.832,0.832,0.831
2,BERT (7),0.859,0.859,0.859,0.859
3,RoBERTa (3),0.766,0.766,0.766,0.764
4,RoBERTa (5),0.834,0.834,0.834,0.833
5,RoBERTa (7),0.829,0.829,0.829,0.827
6,DeBERTa (3),0.846,0.847,0.846,0.845
7,DeBERTa (5),0.830,0.830,0.830,0.829
8,DeBERTa (7),0.823,0.823,0.823,0.823
9,DistilBERT (3),0.848,0.848,0.848,0.848


BEST MODEL

In [ ]:
best_model = df_results.loc[df_results['F1 Score'].idxmax()]
print("Best Model:\n", best_model)

Best Model:
, Model        BERT (7)
,Accuracy     0.858929
,Precision    0.858696
,Recall       0.858929
,F1 Score     0.858742
,Name: 2, dtype: object
